# Verif_aplikasi: pemeriksaan keluaran aplikasi dan pemilihan 50 sampel uji

## 1. Setup

In [6]:
import os
import sys
import json
import shutil
import time
import warnings
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42

SKRIPSI_DIR = Path.home() / "Documents" / "Skripsi"
DATASET_PATH = SKRIPSI_DIR / "plantvillage_tomato_subset"
MODEL_DIR = SKRIPSI_DIR / "SVM_bertahap"        # model dan komponen pendukung (hanya dibaca)
KELAS_LAIN_DIR = SKRIPSI_DIR / "plantvillage_kelas_lain" / "Tomato___Target_Spot"
VERIF_LAMA_DIR = MODEL_DIR / "Verifikasi_UI"    # hasil UI_Verifikasi (U01-U14 tervalidasi, hanya dibaca)
OUTPUT_DIR = SKRIPSI_DIR / "Verif_aplikasi"     # folder keluaran baru
VERIF_DIR = OUTPUT_DIR
VERIF_DIR.mkdir(parents=True, exist_ok=True)

wajib = ["feature_utils.py", "feature_config.json", "svm_m1_raw.pkl", "svm_m2_raw_glcm.pkl",
         "svm_m3_raw_glcm_hog_lbp.pkl", "raw_scaler.pkl", "glcm_scaler.pkl", "hog_scaler.pkl", "lbp_scaler.pkl"]
hilang = [n for n in wajib if not (MODEL_DIR / n).exists()]
if hilang:
    raise FileNotFoundError(f"Berkas berikut tidak ditemukan di {MODEL_DIR}: {hilang}")

sys.path.insert(0, str(MODEL_DIR))
from feature_utils import (preprocess_image, extract_raw_pixel_features,
                           extract_glcm_features, extract_hog_features, extract_lbp_features)

with open(MODEL_DIR / "feature_config.json", encoding="utf-8") as f:
    cfg = json.load(f)

CLASSES = cfg["classes"]
CLASS_LABELS = cfg["class_names"]

models = {
    "M1_Raw": joblib.load(MODEL_DIR / "svm_m1_raw.pkl"),
    "M2_Raw_GLCM": joblib.load(MODEL_DIR / "svm_m2_raw_glcm.pkl"),
    "M3_Raw_GLCM_HOG_LBP": joblib.load(MODEL_DIR / "svm_m3_raw_glcm_hog_lbp.pkl"),
}
nama_model = {"M1_Raw": "M1", "M2_Raw_GLCM": "M2", "M3_Raw_GLCM_HOG_LBP": "M3"}
scalers = {k: joblib.load(MODEL_DIR / f"{k}_scaler.pkl") for k in ["raw", "glcm", "hog", "lbp"]}

print("Model dan komponen pendukung dimuat")
print("Folder keluaran:", VERIF_DIR)
print("Sampel tervalidasi (U01-U14):", VERIF_LAMA_DIR / "sampel_ui.csv",
      "| ditemukan" if (VERIF_LAMA_DIR / "sampel_ui.csv").exists() else "| TIDAK ditemukan")

Model dan komponen pendukung dimuat
Folder keluaran: C:\Users\lenovo\Documents\Skripsi\Verif_aplikasi
Sampel tervalidasi (U01-U14): C:\Users\lenovo\Documents\Skripsi\SVM_bertahap\Verifikasi_UI\sampel_ui.csv | TIDAK ditemukan


## 2. Rekonstruksi pembagian data (urutan dan seed yang sama)

In [7]:
paths, labels = [], []
for cls_idx, cls_name in enumerate(CLASSES):
    for img_file in (DATASET_PATH / cls_name).iterdir():
        if img_file.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue
        if cv2.imread(str(img_file)) is None:
            continue
        paths.append(str(img_file))
        labels.append(cls_idx)

y = np.array(labels, dtype=np.int32)
idx_train, idx_test = train_test_split(
    np.arange(len(paths)), test_size=0.20, stratify=y, random_state=SEED)

print(len(idx_train), len(idx_test))   # harus 5016 dan 1255
if (len(idx_train), len(idx_test)) != (5016, 1255):
    print("PERINGATAN: jumlah data latih/uji tidak sesuai penelitian (5016 dan 1255). Periksa dataset.")

daftar_uji = pd.DataFrame({"path": [paths[i] for i in idx_test],
                           "file": [Path(paths[i]).name for i in idx_test],
                           "label": [CLASS_LABELS[y[i]] for i in idx_test]})
daftar_latih = pd.DataFrame({"path": [paths[i] for i in idx_train],
                             "file": [Path(paths[i]).name for i in idx_train],
                             "label": [CLASS_LABELS[y[i]] for i in idx_train]})
daftar_uji.to_csv(VERIF_DIR / "daftar_data_uji.csv", index=False)
daftar_latih.to_csv(VERIF_DIR / "daftar_data_latih.csv", index=False)
print("Daftar data latih dan uji tersimpan di", VERIF_DIR)

5016 1255
Daftar data latih dan uji tersimpan di C:\Users\lenovo\Documents\Skripsi\Verif_aplikasi


## 3. Ekstraksi fitur dengan fungsi yang sama seperti aplikasi

In [8]:
def ekstrak(path_list):
    raw, glcm, hog_, lbp = [], [], [], []
    for p in path_list:
        img = preprocess_image(cv2.imread(p), cfg["img_size"])
        raw.append(extract_raw_pixel_features(img, cfg["raw_pixel_size"]))
        glcm.append(extract_glcm_features(img, cfg["glcm_distances"], cfg["glcm_angles"], cfg["glcm_props"]))
        hog_.append(extract_hog_features(img, cfg["hog_orientations"],
                                         cfg["hog_pixels_per_cell"], cfg["hog_cells_per_block"]))
        lbp.append(extract_lbp_features(img, cfg["lbp_p"], cfg["lbp_r"]))
    raw = scalers["raw"].transform(np.vstack(raw))
    glcm = scalers["glcm"].transform(np.vstack(glcm))
    hog_ = scalers["hog"].transform(np.vstack(hog_))
    lbp = scalers["lbp"].transform(np.vstack(lbp))
    return {
        "M1_Raw": raw,
        "M2_Raw_GLCM": np.concatenate([raw, glcm], axis=1),
        "M3_Raw_GLCM_HOG_LBP": np.concatenate([raw, glcm, hog_, lbp], axis=1),
    }


sampel_latih = np.random.RandomState(SEED).choice(idx_train, size=min(1000, len(idx_train)), replace=False)
fitur_uji = ekstrak([paths[i] for i in idx_test])
fitur_latih = ekstrak([paths[i] for i in sampel_latih])
print("Ekstraksi fitur selesai")

Ekstraksi fitur selesai


## 4. Validasi: accuracy uji hasil rekonstruksi harus sama persis dengan hasil *holdout*

In [9]:
y_test = y[idx_test]
for name, clf in models.items():
    acc = accuracy_score(y_test, clf.predict(fitur_uji[name]))
    print(f"{name:24s} akurasi uji rekonstruksi: {acc*100:.2f}%")

M1_Raw                   akurasi uji rekonstruksi: 88.45%
M2_Raw_GLCM              akurasi uji rekonstruksi: 90.36%
M3_Raw_GLCM_HOG_LBP      akurasi uji rekonstruksi: 92.35%


## 5. Perbandingan tingkat keyakinan pada data latih dan data uji

In [10]:
rows = []
for name, clf in models.items():
    p_tr = clf.predict_proba(fitur_latih[name]).max(axis=1)
    p_te = clf.predict_proba(fitur_uji[name]).max(axis=1)
    rows.append({
        "model": name,
        "keyakinan_latih_rata2": round(p_tr.mean(), 4),
        "keyakinan_uji_rata2": round(p_te.mean(), 4),
        "latih_>=_0.99_%": round((p_tr >= 0.99).mean() * 100, 1),
        "uji_>=_0.99_%": round((p_te >= 0.99).mean() * 100, 1),
        "uji_>=_0.60_%": round((p_te >= 0.60).mean() * 100, 1),
    })
hasil = pd.DataFrame(rows)
hasil.to_csv(VERIF_DIR / "perbandingan_keyakinan.csv", index=False)
print(hasil.to_string(index=False))

              model  keyakinan_latih_rata2  keyakinan_uji_rata2  latih_>=_0.99_%  uji_>=_0.99_%  uji_>=_0.60_%
             M1_Raw                 0.9959               0.9125             85.1           50.8           92.7
        M2_Raw_GLCM                 0.9968               0.9260             94.0           55.7           94.2
M3_Raw_GLCM_HOG_LBP                 0.9981               0.9385            100.0           61.2           95.6


## 6. Salin seluruh citra uji ke folder per kelas

In [11]:
df_uji = pd.read_csv(VERIF_DIR / "daftar_data_uji.csv")
for _, r in df_uji.iterrows():
    tujuan = VERIF_DIR / "citra_uji" / r["label"]
    tujuan.mkdir(parents=True, exist_ok=True)
    shutil.copy2(r["path"], tujuan / r["file"])

print(len(df_uji), "citra uji disalin ke", VERIF_DIR / "citra_uji")
print(df_uji["label"].value_counts())

1255 citra uji disalin ke C:\Users\lenovo\Documents\Skripsi\Verif_aplikasi\citra_uji
label
Late Blight     382
Septoria        355
Healthy         318
Early Blight    200
Name: count, dtype: int64


## 7. Prediksi seluruh data uji dan pengelompokan kasus

In [12]:
df_kasus = df_uji.copy()
for name, clf in models.items():
    m = nama_model[name]
    proba = clf.predict_proba(fitur_uji[name])
    df_kasus[f"pred_{m}"] = [CLASS_LABELS[i] for i in clf.predict(fitur_uji[name])]
    df_kasus[f"yakin_{m}"] = proba.max(axis=1).round(4)
    df_kasus[f"benar_{m}"] = df_kasus[f"pred_{m}"] == df_kasus["label"]


def kategori(r):
    b1, b2, b3 = r["benar_M1"], r["benar_M2"], r["benar_M3"]
    if b1 and b2 and b3:
        return "semua_benar"
    if not b1 and b3:
        return "M3_memperbaiki_M1"
    if b1 and not b3:
        return "M3_memperburuk_M1"
    if not (b1 or b2 or b3):
        return "semua_salah"
    return "lainnya"


df_kasus["kategori"] = df_kasus.apply(kategori, axis=1)
df_kasus.to_csv(VERIF_DIR / "prediksi_data_uji.csv", index=False)
print(df_kasus["kategori"].value_counts())
print(pd.crosstab(df_kasus["label"], df_kasus["kategori"]))

kategori
semua_benar          1088
semua_salah            76
M3_memperbaiki_M1      66
M3_memperburuk_M1      17
lainnya                 8
Name: count, dtype: int64
kategori      M3_memperbaiki_M1  M3_memperburuk_M1  lainnya  semua_benar  \
label                                                                      
Early Blight                 16                  6        2          134   
Healthy                      14                  0        1          303   
Late Blight                  23                  8        1          327   
Septoria                     13                  3        4          324   

kategori      semua_salah  
label                      
Early Blight           42  
Healthy                 0  
Late Blight            23  
Septoria               11  


## 8. Pemilihan 14 citra uji U01–U14 (kode sama persis dengan `UI_Verifikasi`)

In [13]:
bagian = []
kelas_urut = sorted(df_kasus["label"].unique())

for lbl in kelas_urut:
    g = df_kasus[(df_kasus["kategori"] == "semua_benar") & (df_kasus["label"] == lbl)]
    bagian.append(g.sample(n=min(2, len(g)), random_state=SEED))

for lbl in kelas_urut:
    g = df_kasus[(df_kasus["kategori"] == "M3_memperbaiki_M1") & (df_kasus["label"] == lbl)]
    bagian.append(g.sample(n=min(1, len(g)), random_state=SEED))

g_salah = df_kasus[df_kasus["kategori"] == "semua_salah"]
bagian.append(g_salah.sample(n=min(2, len(g_salah)), random_state=SEED))

sampel = pd.concat(bagian).reset_index(drop=True)
sampel.insert(0, "kode", [f"U{i+1:02d}" for i in range(len(sampel))])

folder_ui = VERIF_DIR / "sampel_ui"
folder_ui.mkdir(parents=True, exist_ok=True)
for _, r in sampel.iterrows():
    shutil.copy2(r["path"], folder_ui / f'{r["kode"]}_{r["label"].replace(" ", "")}_{r["file"]}')

kolom = ["kode", "file", "label", "kategori", "pred_M1", "yakin_M1",
         "pred_M2", "yakin_M2", "pred_M3", "yakin_M3"]
sampel[kolom].to_csv(VERIF_DIR / "sampel_ui.csv", index=False)
print(sampel[kolom].to_string(index=False))

kode                                                        file        label          kategori      pred_M1  yakin_M1      pred_M2  yakin_M2      pred_M3  yakin_M3
 U01   b7294a44-e1fd-4eab-97ac-ed6669c58d8f___RS_Erly.B 7514.JPG Early Blight       semua_benar Early Blight    0.9993 Early Blight    0.9995 Early Blight    1.0000
 U02   22247648-1650-43d5-8a9b-e03f63cf2f1c___RS_Erly.B 7535.JPG Early Blight       semua_benar Early Blight    0.8994 Early Blight    0.9627 Early Blight    0.9966
 U03   337e35e3-7b0d-4760-a1f6-c26c7db90a0b___GH_HL Leaf 425.JPG      Healthy       semua_benar      Healthy    0.9999      Healthy    0.9999      Healthy    1.0000
 U04   5c4d21aa-76f8-4ab7-8440-65fd3cac640b___GH_HL Leaf 263.JPG      Healthy       semua_benar      Healthy    0.9998      Healthy    0.9999      Healthy    1.0000
 U05   2ccd5383-74a3-41d5-9483-110a20ebf247___RS_Late.B 5329.JPG  Late Blight       semua_benar  Late Blight    0.9983  Late Blight    0.9999  Late Blight    0.9999
 U06  1350

### 8a. Penjaga: U01–U14 harus sama dengan sampel yang sudah diperiksa

In [14]:
FILE_SAMPEL_LAMA = VERIF_LAMA_DIR / "sampel_ui.csv"
sampel_14 = sampel.copy()

if FILE_SAMPEL_LAMA.exists():
    lama14 = pd.read_csv(FILE_SAMPEL_LAMA)
    sama_ukuran = len(lama14) == len(sampel_14)
    sama_teks = sama_ukuran and all(
        (lama14[c].astype(str).values == sampel_14[c].astype(str).values).all()
        for c in ["kode", "file", "label", "kategori", "pred_M1", "pred_M2", "pred_M3"])
    sama_yakin = sama_ukuran and all(
        np.allclose(lama14[f"yakin_{m}"].values, sampel_14[f"yakin_{m}"].values, atol=1e-4)
        for m in ["M1", "M2", "M3"])
    if not (sama_teks and sama_yakin):
        raise RuntimeError(
            "U01-U14 hasil rekonstruksi BERBEDA dengan sampel tervalidasi di " + str(FILE_SAMPEL_LAMA)
            + ". Proses dihentikan agar sampel tervalidasi tidak tergantikan. Periksa dataset dan model.")
    print("U01-U14 identik dengan sampel tervalidasi (kode, file, label, kategori, prediksi, keyakinan).")
else:
    print("PERINGATAN: sampel tervalidasi tidak ditemukan di", FILE_SAMPEL_LAMA)
    print("U01-U14 tetap dipilih dengan kode yang sama, tetapi kesamaannya belum dapat dibuktikan.")

sampel_14[kolom].to_csv(VERIF_DIR / "sampel_ui_U01_U14.csv", index=False)

PERINGATAN: sampel tervalidasi tidak ditemukan di C:\Users\lenovo\Documents\Skripsi\SVM_bertahap\Verifikasi_UI\sampel_ui.csv
U01-U14 tetap dipilih dengan kode yang sama, tetapi kesamaannya belum dapat dibuktikan.


### 8b. Penambahan 36 citra uji U15–U50

Penambahan diambil dari data uji yang **belum** terpilih sebagai U01–U14, dengan `random_state` yang sama (seed 42):
1. **Skenario tambahan (U15–U18):** dua citra yang benar pada M1 tetapi salah pada M3 (`M3_memperburuk_M1`) dan dua citra
   yang hanya M2 berbeda sendiri (`lainnya`). Kedua skenario ini belum terwakili pada U01–U14, padahal keduanya terjadi pada
   data uji, sehingga demo tidak hanya menampilkan kasus yang menguntungkan M3.
2. **Sampel representatif (U19–U50):** delapan citra per kelas yang diambil acak dari sisa data uji, sehingga kategori
   benar dan salah muncul sesuai kecenderungan alaminya.

Seperti U01–U14, ke-50 citra ini adalah sampel pengujian fungsional aplikasi dan **tidak** dipakai untuk menghitung accuracy.

In [ ]:
N_TARGET = 50
N_SKENARIO = 2   

sisa = df_kasus[~df_kasus["path"].isin(sampel_14["path"])]

bagian_skenario = []
for kat in ["M3_memperburuk_M1", "lainnya"]:
    g = sisa[sisa["kategori"] == kat]
    bagian_skenario.append(g.sample(n=min(N_SKENARIO, len(g)), random_state=SEED))
skenario = pd.concat(bagian_skenario)
skenario["kelompok"] = "Tambahan: skenario"
sisa = sisa[~sisa["path"].isin(skenario["path"])]

kebutuhan = N_TARGET - len(sampel_14) - len(skenario)
per_kelas, lebih = divmod(kebutuhan, len(kelas_urut))
bagian_rep = []
for j, lbl in enumerate(kelas_urut):
    g = sisa[sisa["label"] == lbl]
    bagian_rep.append(g.sample(n=min(per_kelas + (1 if j < lebih else 0), len(g)), random_state=SEED))
representatif = pd.concat(bagian_rep)
sisa = sisa[~sisa["path"].isin(representatif["path"])]

kurang = kebutuhan - len(representatif)
if kurang > 0:  
    representatif = pd.concat([representatif, sisa.sample(n=min(kurang, len(sisa)), random_state=SEED)])
representatif["kelompok"] = "Tambahan: representatif"

tambahan = pd.concat([skenario, representatif]).reset_index(drop=True)
tambahan.insert(0, "kode", [f"U{len(sampel_14) + i + 1:02d}" for i in range(len(tambahan))])

sampel_14["kelompok"] = "U01-U14 tervalidasi"
sampel_50 = pd.concat([sampel_14, tambahan]).reset_index(drop=True)

assert sampel_50["path"].is_unique, "Ada citra ganda pada sampel"
assert set(sampel_50["path"]).issubset(set(daftar_uji["path"])), "Ada citra di luar data uji"
assert set(sampel_50["path"]).isdisjoint(set(daftar_latih["path"])), "Ada citra latih pada sampel"
assert (sampel_50.iloc[:len(sampel_14)]["file"].values == sampel_14["file"].values).all()

for _, r in tambahan.iterrows():
    shutil.copy2(r["path"], folder_ui / f'{r["kode"]}_{r["label"].replace(" ", "")}_{r["file"]}')

kolom_50 = ["kode", "kelompok"] + kolom[1:]
sampel_50[kolom_50].to_csv(VERIF_DIR / "sampel_ui.csv", index=False)

print(f"Jumlah sampel: {len(sampel_50)} citra ({len(sampel_14)} tervalidasi + {len(tambahan)} tambahan)")
print(f"Jumlah berkas di {folder_ui}: {len(list(folder_ui.iterdir()))}")
print(tambahan[["kode", "kelompok"] + kolom[1:]].to_string(index=False))
print()
print(pd.crosstab(sampel_50["label"], sampel_50["kategori"], margins=True, margins_name="Total"))

Jumlah sampel: 50 citra (14 tervalidasi + 36 tambahan)
Jumlah berkas di C:\Users\lenovo\Documents\Skripsi\Verif_aplikasi\sampel_ui: 50
kode                kelompok                                                         file        label          kategori      pred_M1  yakin_M1      pred_M2  yakin_M2      pred_M3  yakin_M3
 U15      Tambahan: skenario    7b2a8123-3905-48d9-be99-b5e749b5ba45___RS_Erly.B 6472.JPG Early Blight M3_memperburuk_M1 Early Blight    0.8215 Early Blight    0.6271     Septoria    0.5497
 U16      Tambahan: skenario    11922494-321b-4742-afbe-f6db6a8b4005___RS_Erly.B 9392.JPG Early Blight M3_memperburuk_M1 Early Blight    0.7651 Early Blight    0.8229  Late Blight    0.5146
 U17      Tambahan: skenario    bdf7f885-1619-4b3f-bba8-e97896af0de5___RS_Erly.B 7819.JPG Early Blight           lainnya Early Blight    0.5416  Late Blight    0.4765 Early Blight    0.9585
 U18      Tambahan: skenario  07679a50-9d8d-4612-80f4-464785781d56___JR_Sept.L.S 2727.JPG     Septoria   

## 9. Pemeriksaan penolakan pada kelas di luar cakupan (Target Spot)

In [16]:
if KELAS_LAIN_DIR.exists():
    ps = sorted(str(p) for p in KELAS_LAIN_DIR.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"])
    ps = list(np.random.RandomState(SEED).choice(ps, size=min(200, len(ps)), replace=False))
    fitur_lain = ekstrak(ps)

    baris = []
    for name, clf in models.items():
        maks = clf.predict_proba(fitur_lain[name]).max(axis=1)
        for t in [0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
            baris.append({"ambang": t, "model": nama_model[name], "ditolak_%": round((maks < t).mean() * 100, 1)})
    tabel_tolak = pd.DataFrame(baris).pivot(index="ambang", columns="model", values="ditolak_%")
    tabel_tolak.to_csv(VERIF_DIR / "penolakan_target_spot.csv")
    print(f"Jumlah citra Target Spot yang diperiksa: {len(ps)}")
    print(tabel_tolak)
else:
    print("Folder Target Spot tidak ditemukan, sel ini dilewati:", KELAS_LAIN_DIR)

Jumlah citra Target Spot yang diperiksa: 200
model     M1    M2    M3
ambang                  
0.4      3.0   2.0   4.0
0.5     11.0  10.5  12.5
0.6     25.5  23.5  24.0
0.7     37.0  38.0  35.5
0.8     55.5  51.5  50.0
0.9     70.5  70.0  68.5


## 10. Analisis ambang keyakinan pada data uji

Analisis deskriptif sensitivitas ambang. Akurasi di sini dihitung dari `argmax` probabilitas (label yang ditampilkan
aplikasi), sehingga dapat berbeda sangat tipis dari accuracy pada Bab V yang memakai `predict()`. 

In [17]:
baris = []
for t in [0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    for name, clf in models.items():
        p = clf.predict_proba(fitur_uji[name])
        terima = p.max(axis=1) >= t
        akurasi = (p.argmax(axis=1)[terima] == y_test[terima]).mean()
        baris.append({"ambang": t, "model": nama_model[name],
                      "cakupan_%": round(terima.mean() * 100, 1),
                      "akurasi_diterima_%": round(akurasi * 100, 2)})
tabel_ambang = pd.DataFrame(baris).pivot(index="ambang", columns="model")
tabel_ambang.to_csv(VERIF_DIR / "analisis_ambang.csv")
print(tabel_ambang)

       cakupan_%             akurasi_diterima_%              
model         M1    M2    M3                 M1     M2     M3
ambang                                                       
0.4         99.6  99.2  99.7              88.88  91.16  92.65
0.5         97.5  97.8  98.5              90.11  91.86  93.12
0.6         92.7  94.2  95.6              92.10  93.32  94.67
0.7         88.7  91.0  91.6              93.71  94.31  95.91
0.8         83.5  86.5  88.4              94.47  95.40  97.02
0.9         73.7  77.3  82.4              96.65  97.42  97.49


## 11. Lembar acuan: perhitungan terpisah dari aplikasi untuk 50 citra uji

Label acuan memakai `argmax` dari `predict_proba`, sama dengan cara aplikasi menentukan label. Kolom `tampil_*` menambahkan
aturan ambang aplikasi (0,60): bila keyakinan tertinggi di bawah ambang, aplikasi menampilkan **"Tidak dikenali"**. Kolom
ini diperlukan agar citra seperti U10, U12, dan U14 pada M1 tidak terhitung sebagai ketidakcocokan semu.

In [18]:
DESIMAL_TAMPIL = 2
AMBANG_APLIKASI = 0.60
FILE_LEMBAR = VERIF_DIR / "lembar_pencocokan.csv"
FILE_LEMBAR_LAMA = VERIF_LAMA_DIR / "lembar_pencocokan.csv"

sudah_terisi = False
if FILE_LEMBAR.exists():
    lama = pd.read_csv(FILE_LEMBAR)
    kolom_ui = [c for c in lama.columns if c.startswith("ui_")]
    sudah_terisi = bool(kolom_ui) and lama[kolom_ui].notna().any().any()

if sudah_terisi:
    print("Lembar sudah berisi isian, tidak ditimpa:", FILE_LEMBAR)
else:
    ref = sampel_50[["kode", "kelompok", "file", "label"]].copy()
    fitur_sampel = ekstrak(sampel_50["path"].tolist())
    for name, clf in models.items():
        m = nama_model[name]
        proba = clf.predict_proba(fitur_sampel[name])
        maks = proba.max(axis=1)
        ref[f"label_{m}"] = [CLASS_LABELS[i] for i in proba.argmax(axis=1)]
        ref[f"tampil_{m}"] = np.where(maks >= AMBANG_APLIKASI, ref[f"label_{m}"], "Tidak dikenali")
        ref[f"yakin_{m}"] = (maks * 100).round(DESIMAL_TAMPIL)
        for j, c in enumerate(CLASS_LABELS):
            ref[f"p_{m}_{c.replace(' ', '')}"] = (proba[:, j] * 100).round(DESIMAL_TAMPIL)
        ref[f"ui_label_{m}"] = ""
        ref[f"ui_yakin_{m}"] = ""

    if FILE_LEMBAR_LAMA.exists():
        lama = pd.read_csv(FILE_LEMBAR_LAMA, dtype=str).set_index("kode")
        ref = ref.set_index("kode")
        dibawa = 0
        for c in [c for c in lama.columns if c.startswith("ui_") and c in ref.columns]:
            isi_lama = lama[c].dropna()
            isi_lama = isi_lama[isi_lama.str.strip() != ""]
            ref.loc[isi_lama.index, c] = isi_lama
            dibawa += len(isi_lama)
        ref = ref.reset_index()
        print(f"Isian manual dari lembar lama yang dibawa: {dibawa} sel")

    ref.to_csv(FILE_LEMBAR, index=False)
    print("Lembar acuan tersimpan di", FILE_LEMBAR)
    print(ref[["kode", "tampil_M1", "yakin_M1", "tampil_M2", "yakin_M2", "tampil_M3", "yakin_M3"]].to_string(index=False))

Lembar acuan tersimpan di C:\Users\lenovo\Documents\Skripsi\Verif_aplikasi\lembar_pencocokan.csv
kode      tampil_M1  yakin_M1      tampil_M2  yakin_M2      tampil_M3  yakin_M3
 U01   Early Blight     99.93   Early Blight     99.95   Early Blight    100.00
 U02   Early Blight     89.94   Early Blight     96.27   Early Blight     99.66
 U03        Healthy     99.99        Healthy     99.99        Healthy    100.00
 U04        Healthy     99.98        Healthy     99.99        Healthy    100.00
 U05    Late Blight     99.83    Late Blight     99.99    Late Blight     99.99
 U06    Late Blight    100.00    Late Blight    100.00    Late Blight     99.13
 U07       Septoria     99.82       Septoria    100.00       Septoria    100.00
 U08       Septoria    100.00       Septoria    100.00       Septoria    100.00
 U09        Healthy     96.94   Early Blight     70.47   Early Blight     89.14
 U10 Tidak dikenali     46.05        Healthy     99.98        Healthy     99.99
 U11   Early Blight    

## 12. Pencocokan keluaran aplikasi dengan perhitungan terpisah

Ada dua cara mengisi kolom `ui_label_*` dan `ui_yakin_*`:

**Cara A — otomatis dari aplikasi (disarankan untuk 50 citra)**
1. Jalankan aplikasi `Aplikasi_perbandingan`.
2. Unggah seluruh berkas di `Verif_aplikasi/sampel_ui` (U01–U50) sekaligus, lalu jalankan klasifikasi.
3. Tekan **Unduh hasil (CSV)** dan simpan berkas `hasil_aplikasi_batch.csv` ke folder `Verif_aplikasi`.
4. Simpan tangkapan layar ringkasan, tabel hasil, dan beberapa detail citra untuk lampiran.

**Cara B — manual** (seperti `UI_Verifikasi`): catat label dan keyakinan per citra ke `lembar_pencocokan.csv`. Bacalah
keyakinan dari tabel probabilitas (dua desimal), bukan dari kartu metrik yang hanya menampilkan satu desimal.

Bila `hasil_aplikasi_batch.csv` ada, isiannya dipakai untuk kode yang tercantum di berkas tersebut. Label UI dinyatakan cocok
bila sama dengan `tampil_*` (aturan ambang) atau dengan `label_*` (argmax pada tabel probabilitas).

In [ ]:
FILE_EKSPOR_UI = VERIF_DIR / "hasil_aplikasi_batch.csv"
isi = pd.read_csv(FILE_LEMBAR, dtype={f"ui_label_{m}": str for m in ["M1", "M2", "M3"]})

if FILE_EKSPOR_UI.exists():
    eks = pd.read_csv(FILE_EKSPOR_UI)
    eks["kode"] = eks["nama_berkas"].astype(str).str.split("_").str[0]
    eks = eks[eks["kode"].isin(isi["kode"])].drop_duplicates("kode").set_index("kode")
    isi = isi.set_index("kode")
    for m in ["M1", "M2", "M3"]:
        isi[f"ui_label_{m}"] = isi[f"ui_label_{m}"].astype(object)
        isi.loc[eks.index, f"ui_label_{m}"] = eks[f"pred_{m}"]
        isi.loc[eks.index, f"ui_yakin_{m}"] = eks[f"yakin_{m}"]
    isi = isi.reset_index()
    hilang_ui = sorted(set(isi["kode"]) - set(eks.index))
    print(f"Isian diambil dari {FILE_EKSPOR_UI.name}: {len(eks)} citra")
    if hilang_ui:
        print("Kode yang tidak ada pada ekspor aplikasi:", hilang_ui)

kolom_ui = [c for c in isi.columns if c.startswith("ui_")]
if isi[kolom_ui].isna().all().all():
    print("Kolom ui_* masih kosong. Gunakan Cara A atau Cara B, lalu jalankan ulang sel ini.")
else:
    TOL = 0.5 * 10 ** (-DESIMAL_TAMPIL)   
    for m in ["M1", "M2", "M3"]:
        ui_lbl = isi[f"ui_label_{m}"].astype(str).str.strip()
        isi[f"cocok_label_{m}"] = (ui_lbl == isi[f"tampil_{m}"].astype(str).str.strip()) | \
                                  (ui_lbl == isi[f"label_{m}"].astype(str).str.strip())
        isi[f"selisih_yakin_{m}"] = (isi[f"yakin_{m}"] - pd.to_numeric(isi[f"ui_yakin_{m}"], errors="coerce")).abs()
        isi[f"cocok_yakin_{m}"] = isi[f"selisih_yakin_{m}"] <= TOL
    cek = [c for c in isi.columns if c.startswith("cocok_")]
    isi["cocok_semua"] = isi[cek].all(axis=1)
    print("Semua cocok:", bool(isi["cocok_semua"].all()))
    print("Jumlah cocok per pemeriksaan (dari", len(isi), "citra):")
    print(isi[cek].sum())
    print()
    print("Rekap per kelompok sampel:")
    print(isi.groupby("kelompok", sort=False)["cocok_semua"].agg(jumlah="size", cocok="sum"))
    tidak_cocok = isi[~isi["cocok_semua"]]
    if len(tidak_cocok):
        print()
        print("Citra yang belum cocok:")
        print(tidak_cocok[["kode"] + [c for c in isi.columns if c.startswith("selisih_")]].round(3).to_string(index=False))
    isi.to_csv(VERIF_DIR / "hasil_pencocokan.csv", index=False)
    print("Hasil pencocokan tersimpan di", VERIF_DIR / "hasil_pencocokan.csv")

Isian diambil dari hasil_aplikasi_batch.csv: 50 citra
Semua cocok: True
Jumlah cocok per pemeriksaan (dari 50 citra):
cocok_label_M1    50
cocok_yakin_M1    50
cocok_label_M2    50
cocok_yakin_M2    50
cocok_label_M3    50
cocok_yakin_M3    50
dtype: int64

Rekap per kelompok sampel:
                         jumlah  cocok
kelompok                              
U01-U14 tervalidasi          14     14
Tambahan: skenario            4      4
Tambahan: representatif      32     32
Hasil pencocokan tersimpan di C:\Users\lenovo\Documents\Skripsi\Verif_aplikasi\hasil_pencocokan.csv


## 13. Uji beban: waktu proses 50 citra dengan alur yang sama seperti aplikasi

Sel ini mengukur waktu yang dibutuhkan untuk memproses seluruh berkas di `sampel_ui` **satu per satu**, sama seperti
aplikasi: berkas dibaca sebagai *bytes*, didekode dengan `cv2.imdecode`, lalu setiap model menjalankan
praproses, ekstraksi fitur, normalisasi, dan `predict_proba`. Hasilnya menjadi dasar empiris bahwa batas 50 citra aman
dijalankan pada perangkat yang dipakai untuk demo.

In [22]:
def proses_seperti_aplikasi(image_bytes):
    image_bgr = cv2.imdecode(np.frombuffer(image_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
    hasil = {}
    for name, clf in models.items():
        img = preprocess_image(image_bgr, cfg["img_size"])
        fitur = [scalers["raw"].transform(extract_raw_pixel_features(img, cfg["raw_pixel_size"]))]
        if name != "M1_Raw":
            fitur.append(scalers["glcm"].transform(
                extract_glcm_features(img, cfg["glcm_distances"], cfg["glcm_angles"], cfg["glcm_props"])))
        if name == "M3_Raw_GLCM_HOG_LBP":
            fitur.append(scalers["hog"].transform(extract_hog_features(
                img, cfg["hog_orientations"], cfg["hog_pixels_per_cell"], cfg["hog_cells_per_block"])))
            fitur.append(scalers["lbp"].transform(extract_lbp_features(img, cfg["lbp_p"], cfg["lbp_r"])))
        hasil[nama_model[name]] = clf.predict_proba(np.concatenate(fitur, axis=1))[0]
    return hasil


berkas_ui = sorted(folder_ui.iterdir())
data_bytes = [p.read_bytes() for p in berkas_ui]

waktu = []
for b in data_bytes:
    t0 = time.perf_counter()
    proses_seperti_aplikasi(b)
    waktu.append(time.perf_counter() - t0)
waktu = np.array(waktu)

beban = pd.DataFrame([{
    "jumlah_citra": len(berkas_ui),
    "total_ukuran_KB": round(sum(len(b) for b in data_bytes) / 1024, 1),
    "total_waktu_detik": round(waktu.sum(), 2),
    "rata2_per_citra_detik": round(waktu.mean(), 3),
    "maks_per_citra_detik": round(waktu.max(), 3),
}])
beban.to_csv(VERIF_DIR / "uji_beban_50_citra.csv", index=False)
print(beban.to_string(index=False))

 jumlah_citra  total_ukuran_KB  total_waktu_detik  rata2_per_citra_detik  maks_per_citra_detik
           50            806.3               4.25                  0.085                 1.359
